# Homework 5: Deploying Machine Learning Models


---


## Question 1 — Environment Check

Question: Run uv --version and record the version.

In [1]:
import subprocess

result = subprocess.run(
    ["uv", "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout.strip())


uv 0.13.5 (x86_64-unknown-linux-gnu)


**Answer:** uv 0.13.5


---


## Question 2 — Locked Dependency

Question: Which Scikit-Learn version is part of the 2026 reference environment?

Options:

1. 1.6.1
2. 1.7.2
3. 1.8.0
4. 2.0.0

In [2]:
from pathlib import Path
import re

deployment_dir = Path(
    "/home/dell/ml-zoomcamp/ml-zoomcamp-course/"
    "cohorts/2026/homework/05-deployment"
)

lockfile = deployment_dir / "uv.lock"
lock_text = lockfile.read_text()

match = re.search(
    r'(?ms)^\[\[package\]\]\n'
    r'name = "scikit-learn"\n'
    r'version = "([^"]+)"',
    lock_text,
)

if match is None:
    raise ValueError("Scikit-Learn entry not found in uv.lock")

sklearn_version = match.group(1)
print("Scikit-Learn version:", sklearn_version)

Scikit-Learn version: 1.7.2


**Answer:** Option 2 — 1.7.2


---


## Question 3 — Load the Model

Question: Load pipeline.bin using pickle and call predict_proba for the following lead.

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

Report the probability of conversion rounded to three decimal places.

In [3]:
import pickle
from pathlib import Path

model_path = (
    Path.home()
    / "ml-zoomcamp"
    / "ml-zoomcamp-course"
    / "cohorts"
    / "2026"
    / "homework"
    / "05-deployment"
    / "pipeline.bin"
)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with model_path.open("rb") as f:
    model = pickle.load(f)

probabilities = model.predict_proba([lead])

positive_class_index = list(model.classes_).index(1)
conversion_probability = probabilities[0][positive_class_index]

print("Model classes:", model.classes_)
print("Raw conversion probability:", conversion_probability)
print(f"Conversion probability: {conversion_probability:.3f}")


Model classes: [0 1]
Raw conversion probability: 0.5329944186432507
Conversion probability: 0.533


**Answer:** 0.533


---


## Question 4 — Serve the Model

Question: Start the provided API and send the following lead to the POST /predict endpoint.

```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

Report conversion_probability rounded to three decimal places. The response must also contain a probability between 0 and 1 and a boolean conversion field.

> Start the server in a terminal first:
```bash
cd ~/ml-zoomcamp/ml-zoomcamp-course/cohorts/2026/homework/05-deployment
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

In [5]:
import requests

api_url = "http://localhost:9696"

lead_api = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

response = requests.post(
    f"{api_url}/predict",
    json=lead_api,
    timeout=30,
)

response.raise_for_status()
result = response.json()

print(result)

assert 0 <= result["conversion_probability"] <= 1
assert isinstance(result["conversion"], bool)

print(
    "Conversion probability:",
    f'{result["conversion_probability"]:.3f}',
)

{'conversion_probability': 0.769799, 'conversion': True}
Conversion probability: 0.770


**Answer:** 0.770


---


## Question 5 — Container Configuration

Question: Which Python base-image tag is declared in the Dockerfile?

Options:

1. python:3.9-slim-bullseye
2. python:3.11.15-slim-bookworm
3. python:3.12-slim-bookworm
4. python:3.13.10-slim-bookworm

In [6]:
from pathlib import Path

dockerfile_path = (
    Path.home()
    / "ml-zoomcamp"
    / "ml-zoomcamp-course"
    / "cohorts"
    / "2026"
    / "homework"
    / "05-deployment"
    / "Dockerfile"
)

dockerfile_text = dockerfile_path.read_text()

from_line = next(
    line.strip()
    for line in dockerfile_text.splitlines()
    if line.strip().startswith("FROM ")
)

print("Docker base image declaration:")
print(from_line)

Docker base image declaration:
FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3


**Answer:** Option 2 — python:3.11.15-slim-bookworm


---


## Question 6 — Run the Container

Question: Run the Docker container and send the same request from Question 4. Report conversion_probability rounded to three decimal places.

Also verify that the health endpoint returns status ok and the model checksum recorded in model_metadata.json.

The container was built and run from the official deployment directory using:

```bash
docker build -t zoomcamp-model:2026-hw5 .

docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```
The provided test client was executed with:
```bash
uv run python q6_test.py
```

Observed test output:
```python
{
    'conversion_probability': 0.769799,
    'conversion': True
}
```

The health endpoint was checked with:
```bash
curl -s http://localhost:9696/health
```

Observed response:

```json
{
  "status": "ok",
  "model_sha256": "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1",
  "feature_defaults_sha256": "1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"
}
```

**Answer:** 0.770


---